In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs("../data/processed", exist_ok=True)

nav = pd.read_csv("../data/raw/nav_history.csv")
print("Shape:", nav.shape)
print(nav.dtypes)
print(nav.head())

Shape: (46000, 3)
amfi_code      int64
date             str
nav          float64
dtype: object
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [2]:
# Step 1 — Parse date to datetime
nav["date"] = pd.to_datetime(nav["date"])

# Step 2 — Sort by amfi_code + date
nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

# Step 3 — Remove duplicates
before = len(nav)
nav = nav.drop_duplicates(subset=["amfi_code", "date"])
print(f"Duplicates removed: {before - len(nav)}")

# Step 4 — Forward-fill missing NAV per fund (holidays/weekends)
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()

# Step 5 — Validate NAV > 0
invalid = nav[nav["nav"] <= 0]
print(f"Invalid NAV rows (<=0): {len(invalid)}")
nav = nav[nav["nav"] > 0]

# Step 6 — Final check
print("\nCleaned nav_history shape:", nav.shape)
print(nav.isnull().sum())
print(nav.head())

Duplicates removed: 0
Invalid NAV rows (<=0): 0

Cleaned nav_history shape: (46000, 3)
amfi_code    0
date         0
nav          0
dtype: int64
   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639


In [3]:
nav.to_csv("../data/processed/clean_nav.csv", index=False)
print("Cleaned data saved → data/processed/clean_nav.csv")

Cleaned data saved → data/processed/clean_nav.csv


In [4]:
txn = pd.read_csv("../data/raw/investor_transactions.csv")
print("Shape:", txn.shape)
print(txn.dtypes)
print(txn.head())

Shape: (32778, 13)
investor_id               str
transaction_date          str
amfi_code               int64
transaction_type          str
amount_inr              int64
state                     str
city                      str
city_tier                 str
age_group                 str
gender                    str
annual_income_lakh    float64
payment_mode              str
kyc_status                str
dtype: object
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female  

In [5]:
import re

# Check current unique values
print("transaction_type values:", txn["transaction_type"].unique())
print("kyc_status values:", txn["kyc_status"].unique())
print("city_tier values:", txn["city_tier"].unique())

# Standardise transaction_type — strip spaces, fix capitalisation
txn["transaction_type"] = txn["transaction_type"].str.strip().str.title()
print("\nAfter standardising:", txn["transaction_type"].unique())

transaction_type values: <StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
kyc_status values: <StringArray>
['Verified', 'Pending']
Length: 2, dtype: str
city_tier values: <StringArray>
['T30', 'B30']
Length: 2, dtype: str

After standardising: <StringArray>
['Sip', 'Redemption', 'Lumpsum']
Length: 3, dtype: str


In [6]:
# Fix date to datetime
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"])

# Validate amount > 0
invalid_amount = txn[txn["amount_inr"] <= 0]
print(f"Invalid amount rows (<=0): {len(invalid_amount)}")
txn = txn[txn["amount_inr"] > 0]

print(f"Shape after amount filter: {txn.shape}")

Invalid amount rows (<=0): 0
Shape after amount filter: (32778, 13)


In [7]:
# KYC status check
print("KYC status values:", txn["kyc_status"].unique())
txn["kyc_status"] = txn["kyc_status"].str.strip().str.title()

# Null check
print("\nNull values:")
print(txn.isnull().sum())

# Duplicate check
dupes = txn.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")
txn = txn.drop_duplicates()

print(f"\nFinal shape: {txn.shape}")
print(txn.head())

KYC status values: <StringArray>
['Verified', 'Pending']
Length: 2, dtype: str

Null values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Duplicate rows: 0

Final shape: (32778, 13)
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              Sip        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              Sip         912   
3   INV003436       2024-01-01     118634              Sip        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana 

In [8]:
txn.to_csv("../data/processed/clean_transactions.csv", index=False)
print("Cleaned data Saved → data/processed/clean_transactions.csv")

Cleaned data Saved → data/processed/clean_transactions.csv


In [9]:
perf = pd.read_csv("../data/raw/scheme_performance.csv")
print("Shape:", perf.shape)
print(perf.dtypes)
print(perf.head())

Shape: (40, 19)
amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object
   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4    

In [10]:
return_cols = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct", "benchmark_3yr_pct"]

print("=== RETURN VALUE CHECK ===")
for col in return_cols:
    non_numeric = perf[pd.to_numeric(perf[col], errors="coerce").isnull()]
    print(f"{col} → non-numeric rows: {len(non_numeric)}")

print("\nReturn value ranges:")
print(perf[return_cols].describe())

=== RETURN VALUE CHECK ===
return_1yr_pct → non-numeric rows: 0
return_3yr_pct → non-numeric rows: 0
return_5yr_pct → non-numeric rows: 0
benchmark_3yr_pct → non-numeric rows: 0

Return value ranges:
       return_1yr_pct  return_3yr_pct  return_5yr_pct  benchmark_3yr_pct
count       40.000000       40.000000       40.000000          40.000000
mean        14.376000       14.089000       14.516750          12.835500
std          4.883023        4.617253        4.454021           4.740972
min          4.260000        5.140000        5.430000           3.960000
25%         11.735000       12.035000       12.340000          10.690000
50%         14.620000       14.205000       14.185000          13.090000
75%         16.392500       15.882500       17.585000          14.775000
max         24.930000       23.390000       23.800000          22.160000


In [11]:
print("=== SHARPE RATIO CHECK ===")
negative_sharpe = perf[perf["sharpe_ratio"] < 0]
print(f"Negative Sharpe ratio funds: {len(negative_sharpe)}")

if len(negative_sharpe) > 0:
    print(negative_sharpe[["scheme_name", "sharpe_ratio"]])
else:
    print(" No negative Sharpe ratios found")

print("\nSharpe ratio range:")
print(perf["sharpe_ratio"].describe())

=== SHARPE RATIO CHECK ===
Negative Sharpe ratio funds: 0
 No negative Sharpe ratios found

Sharpe ratio range:
count    40.000000
mean      1.361750
std       1.475805
min       0.800000
25%       0.865000
50%       0.925000
75%       0.985000
max       7.680000
Name: sharpe_ratio, dtype: float64


In [12]:
print("=== EXPENSE RATIO CHECK ===")
out_of_range = perf[
    (perf["expense_ratio_pct"] < 0.1) | 
    (perf["expense_ratio_pct"] > 2.5)
]
print(f"Out of range (0.1% – 2.5%): {len(out_of_range)}")

if len(out_of_range) > 0:
    print(out_of_range[["scheme_name", "expense_ratio_pct"]])
else:
    print("All expense ratios within valid range")

print("\nExpense ratio range:")
print(perf["expense_ratio_pct"].describe())

=== EXPENSE RATIO CHECK ===
Out of range (0.1% – 2.5%): 0
All expense ratios within valid range

Expense ratio range:
count    40.000000
mean      1.237000
std       0.386584
min       0.550000
25%       0.787500
50%       1.425000
75%       1.540000
max       1.640000
Name: expense_ratio_pct, dtype: float64


In [13]:
print("=== NULL CHECK ===")
print(perf.isnull().sum())

print(f"\nDuplicate rows: {perf.duplicated().sum()}")
perf = perf.drop_duplicates()

print(f"\nFinal shape: {perf.shape}")

=== NULL CHECK ===
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

Duplicate rows: 0

Final shape: (40, 19)


In [14]:
perf.to_csv("../data/processed/clean_performance.csv", index=False)
print("Clean data saved → data/processed/clean_performance.csv")

Clean data saved → data/processed/clean_performance.csv


In [15]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\meghr\OneDrive\Desktop\Bluestock\Mutual_funds_Analytis")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"

# ── 1. fund_master ──────────────────────────────────────────────
fund_master = pd.read_csv(RAW / "fund_master.csv")
fund_master.drop_duplicates(inplace=True)
fund_master["launch_date"] = pd.to_datetime(fund_master["launch_date"], errors="coerce")
fund_master["expense_ratio_pct"] = pd.to_numeric(fund_master["expense_ratio_pct"], errors="coerce")
fund_master.to_csv(PROCESSED / "clean_fund_master.csv", index=False)
print(f" clean_fund_master.csv  → {fund_master.shape}")

# ── 2. aum_by_fund_house ────────────────────────────────────────
aum = pd.read_csv(RAW / "aum_by_fund_house.csv")
aum.drop_duplicates(inplace=True)
aum["aum_crore"] = pd.to_numeric(aum["aum_crore"], errors="coerce")
aum.dropna(subset=["aum_crore"], inplace=True)
aum.to_csv(PROCESSED / "clean_aum_by_fund_house.csv", index=False)
print(f" clean_aum_by_fund_house.csv  → {aum.shape}")

# ── 3. monthly_sip_inflows ──────────────────────────────────────
sip = pd.read_csv(RAW / "monthly_sip_inflows.csv")
sip.drop_duplicates(inplace=True)
sip["month"] = pd.to_datetime(sip["month"], format="%Y-%m", errors="coerce")
for col in ["sip_inflow_crore","active_sip_accounts_crore","new_sip_accounts_lakh","sip_aum_lakh_crore","yoy_growth_pct"]:
    if col in sip.columns:
        sip[col] = pd.to_numeric(sip[col], errors="coerce")
sip.to_csv(PROCESSED / "clean_monthly_sip_inflows.csv", index=False)
print(f" clean_monthly_sip_inflows.csv  → {sip.shape}")

# ── 4. category_inflows ─────────────────────────────────────────
cat = pd.read_csv(RAW / "category_inflows.csv")
cat.drop_duplicates(inplace=True)
cat.to_csv(PROCESSED / "clean_category_inflows.csv", index=False)
print(f" clean_category_inflows.csv  → {cat.shape}")

# ── 5. industry_folio_count ─────────────────────────────────────
folio = pd.read_csv(RAW / "industry_folio_count.csv")
folio.drop_duplicates(inplace=True)
folio.to_csv(PROCESSED / "clean_industry_folio_count.csv", index=False)
print(f" clean_industry_folio_count.csv  → {folio.shape}")

# ── 6. portfolio_holdings ───────────────────────────────────────
portfolio = pd.read_csv(RAW / "portfolio_holdings.csv")
portfolio.drop_duplicates(inplace=True)
portfolio["weight_pct"] = pd.to_numeric(portfolio["weight_pct"], errors="coerce")
portfolio.to_csv(PROCESSED / "clean_portfolio_holdings.csv", index=False)
print(f" clean_portfolio_holdings.csv  → {portfolio.shape}")

# ── 7. benchmark_indices ────────────────────────────────────────
bench = pd.read_csv(RAW / "benchmark_indices.csv")
bench.drop_duplicates(inplace=True)
bench["date"] = pd.to_datetime(bench["date"], errors="coerce")
bench.sort_values("date", inplace=True)
bench.ffill(inplace=True)
bench.to_csv(PROCESSED / "clean_benchmark_indices.csv", index=False)
print(f" clean_benchmark_indices.csv  → {bench.shape}")

print("\n All 7 cleaned CSVs saved to data/processed/")

 clean_fund_master.csv  → (40, 15)
 clean_aum_by_fund_house.csv  → (90, 5)
 clean_monthly_sip_inflows.csv  → (48, 6)
 clean_category_inflows.csv  → (144, 3)
 clean_industry_folio_count.csv  → (21, 6)
 clean_portfolio_holdings.csv  → (322, 8)
 clean_benchmark_indices.csv  → (8050, 3)

 All 7 cleaned CSVs saved to data/processed/


In [16]:
import sqlite3
import os
import pandas as pd

db_path = "../data/db/bluestock_mf.db"

# Create folder first
os.makedirs("../data/db", exist_ok=True)
print(" Folder created")

# Delete old db if exists
if os.path.exists(db_path):
    os.remove(db_path)
    print(" Old database deleted")

# Create fresh from schema
with sqlite3.connect(db_path) as conn:
    with open("../sql/schema.sql", "r") as f:
        conn.executescript(f.read())

print(" Schema created — bluestock_mf.db ready")

# Verify tables
with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
    print("\nTables created:")
    print(tables)

 Folder created
 Old database deleted
 Schema created — bluestock_mf.db ready

Tables created:
                name
0           dim_fund
1           fact_nav
2    sqlite_sequence
3  fact_transactions
4   fact_performance
5           fact_aum
6  fact_sip_industry


In [17]:
import pandas as pd
import sqlite3

db_path = "../data/db/bluestock_mf.db"

# Load fund_master → dim_fund
fund_master = pd.read_csv("../data/raw/fund_master.csv")

with sqlite3.connect(db_path) as conn:
    fund_master.to_sql("dim_fund", conn, if_exists="append", index=False)

print(f" dim_fund loaded — {len(fund_master)} rows")

 dim_fund loaded — 40 rows


In [18]:
# Load clean_nav → fact_nav
nav = pd.read_csv("../data/processed/clean_nav.csv")

# Rename date column to match schema
nav = nav.rename(columns={"date": "nav_date"})

with sqlite3.connect(db_path) as conn:
    nav.to_sql("fact_nav", conn, if_exists="append", index=False)

print(f" fact_nav loaded — {len(nav)} rows")

 fact_nav loaded — 46000 rows


In [19]:
# Load clean_transactions → fact_transactions
txn = pd.read_csv("../data/processed/clean_transactions.csv")

# Rename date column to match schema
txn = txn.rename(columns={"transaction_date": "transaction_date"})

with sqlite3.connect(db_path) as conn:
    txn.to_sql("fact_transactions", conn, if_exists="append", index=False)

print(f" fact_transactions loaded — {len(txn)} rows")

 fact_transactions loaded — 32778 rows


In [20]:
perf = pd.read_csv("../data/processed/clean_performance.csv")

# Keep only columns that exist in fact_performance schema
# Also flag negative sharpe ratios
perf["negative_sharpe"] = (perf["sharpe_ratio"] < 0).astype(int)

cols_to_load = [
    "amfi_code", "return_1yr_pct", "return_3yr_pct", "return_5yr_pct",
    "benchmark_3yr_pct", "alpha", "beta", "sharpe_ratio", "sortino_ratio",
    "std_dev_ann_pct", "max_drawdown_pct", "morningstar_rating", "negative_sharpe"
]

perf_db = perf[cols_to_load]

with sqlite3.connect(db_path) as conn:
    perf_db.to_sql("fact_performance", conn, if_exists="append", index=False)

print(f" fact_performance loaded — {len(perf_db)} rows")
print(f"   Negative Sharpe flagged: {perf_db['negative_sharpe'].sum()}")

 fact_performance loaded — 40 rows
   Negative Sharpe flagged: 0


In [21]:
aum = pd.read_csv("../data/raw/aum_by_fund_house.csv")

# Keep only columns that match schema
cols_to_load = ["fund_house", "date", "aum_crore", "num_schemes"]
aum_db = aum[cols_to_load]

with sqlite3.connect(db_path) as conn:
    aum_db.to_sql("fact_aum", conn, if_exists="append", index=False)

print(f" fact_aum loaded — {len(aum_db)} rows")

 fact_aum loaded — 90 rows


In [22]:
sip = pd.read_csv("../data/raw/monthly_sip_inflows.csv")

# Fill NaN in yoy_growth_pct (first year has no prior year to compare)
sip["yoy_growth_pct"] = sip["yoy_growth_pct"].fillna(0)

with sqlite3.connect(db_path) as conn:
    sip.to_sql("fact_sip_industry", conn, if_exists="append", index=False)

print(f"fact_sip_industry loaded — {len(sip)} rows")

fact_sip_industry loaded — 48 rows


In [23]:
tables = ["dim_fund", "fact_nav", "fact_transactions",
          "fact_performance", "fact_aum", "fact_sip_industry"]

with sqlite3.connect(db_path) as conn:
    print("=== ROW COUNTS PER TABLE ===")
    for table in tables:
        count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table}", conn)
        print(f"  {table:<25} {count['count'][0]} rows")

print("\n All data loaded into bluestock_mf.db")

=== ROW COUNTS PER TABLE ===
  dim_fund                  40 rows
  fact_nav                  46000 rows
  fact_transactions         32778 rows
  fact_performance          40 rows
  fact_aum                  90 rows
  fact_sip_industry         48 rows

 All data loaded into bluestock_mf.db


In [24]:
import sqlite3
import pandas as pd

db_path = "../data/db/bluestock_mf.db"

queries = {
    "Q1 Top 5 funds by AUM": """
        SELECT f.fund_house, f.scheme_name, a.aum_crore
        FROM fact_aum a
        JOIN dim_fund f ON a.fund_house = f.fund_house
        ORDER BY a.aum_crore DESC LIMIT 5
    """,
    "Q2 Avg NAV per month": """
        SELECT amfi_code, strftime('%Y-%m', nav_date) AS month,
        ROUND(AVG(nav), 2) AS avg_nav
        FROM fact_nav
        GROUP BY amfi_code, month
        ORDER BY amfi_code, month LIMIT 5
    """,
    "Q3 SIP inflow YoY growth": """
        SELECT month, sip_inflow_crore, ROUND(yoy_growth_pct, 2) AS yoy_growth_pct
        FROM fact_sip_industry
        WHERE yoy_growth_pct > 0
        ORDER BY month LIMIT 5
    """,
    "Q4 Transactions by state": """
        SELECT state, COUNT(*) AS total_transactions,
        ROUND(SUM(amount_inr) / 1000000.0, 2) AS total_amount_millions
        FROM fact_transactions
        GROUP BY state ORDER BY total_transactions DESC
    """,
    "Q5 Funds expense_ratio < 1%": """
        SELECT amfi_code, scheme_name, fund_house, expense_ratio_pct
        FROM dim_fund WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC
    """,
    "Q6 Best 3yr return funds": """
        SELECT f.scheme_name, f.fund_house, p.return_3yr_pct, p.sharpe_ratio
        FROM fact_performance p
        JOIN dim_fund f ON p.amfi_code = f.amfi_code
        ORDER BY p.return_3yr_pct DESC LIMIT 10
    """,
    "Q7 Transaction split by type": """
        SELECT transaction_type, COUNT(*) AS total_count,
        ROUND(SUM(amount_inr) / 1000000.0, 2) AS total_amount_millions,
        ROUND(AVG(amount_inr), 2) AS avg_amount
        FROM fact_transactions
        GROUP BY transaction_type ORDER BY total_count DESC
    """,
    "Q8 Top 5 states by avg SIP": """
        SELECT state, ROUND(AVG(amount_inr), 2) AS avg_sip_amount,
        COUNT(*) AS total_sip_count
        FROM fact_transactions WHERE transaction_type = 'Sip'
        GROUP BY state ORDER BY avg_sip_amount DESC LIMIT 5
    """,
    "Q9 Funds highest alpha": """
        SELECT f.scheme_name, f.fund_house, p.alpha, p.beta, p.return_3yr_pct
        FROM fact_performance p
        JOIN dim_fund f ON p.amfi_code = f.amfi_code
        ORDER BY p.alpha DESC LIMIT 10
    """,
    "Q10 Top 5 SIP inflow months": """
        SELECT month, sip_inflow_crore, active_sip_accounts_crore
        FROM fact_sip_industry
        ORDER BY sip_inflow_crore DESC LIMIT 5
    """
}

with sqlite3.connect(db_path) as conn:
    for name, query in queries.items():
        print(f"\n{'='*55}")
        print(f"  {name}")
        print(f"{'='*55}")
        df = pd.read_sql(query, conn)
        print(df.to_string(index=False))


  Q1 Top 5 funds by AUM
     fund_house                                  scheme_name  aum_crore
SBI Mutual Fund     SBI Bluechip Fund - Direct Plan - Growth  1250000.0
SBI Mutual Fund    SBI Bluechip Fund - Regular Plan - Growth  1250000.0
SBI Mutual Fund SBI Magnum Gilt Fund - Regular Plan - Growth  1250000.0
SBI Mutual Fund    SBI Small Cap Fund - Direct Plan - Growth  1250000.0
SBI Mutual Fund   SBI Small Cap Fund - Regular Plan - Growth  1250000.0

  Q2 Avg NAV per month
 amfi_code   month  avg_nav
    100016 2022-01   512.54
    100016 2022-02   513.93
    100016 2022-03   522.58
    100016 2022-04   525.63
    100016 2022-05   504.31

  Q3 SIP inflow YoY growth
  month  sip_inflow_crore  yoy_growth_pct
2023-01           13856.0           20.31
2023-02           13687.0           19.66
2023-03           14276.0           15.80
2023-04           14749.0           24.33
2023-05           14749.0           20.05

  Q4 Transactions by state
         state  total_transactions  total_a

In [25]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\meghr\OneDrive\Desktop\Bluestock\Mutual_funds_Analytis")
PROCESSED = BASE / "data" / "processed"

datasets = {
    "01_fund_master":           PROCESSED / "clean_fund_master.csv",
    "02_nav_history":           PROCESSED / "clean_nav.csv",
    "03_aum_by_fund_house":     PROCESSED / "clean_aum_by_fund_house.csv",
    "04_monthly_sip_inflows":   PROCESSED / "clean_monthly_sip_inflows.csv",
    "05_category_inflows":      PROCESSED / "clean_category_inflows.csv",
    "06_industry_folio_count":  PROCESSED / "clean_industry_folio_count.csv",
    "07_scheme_performance":    PROCESSED / "clean_performance.csv",
    "08_investor_transactions": PROCESSED / "clean_transactions.csv",
    "09_portfolio_holdings":    PROCESSED / "clean_portfolio_holdings.csv",
    "10_benchmark_indices":     PROCESSED / "clean_benchmark_indices.csv",
}

column_descriptions = {
    "amfi_code": "AMFI unique scheme code (PK / FK)",
    "fund_house": "AMC name (e.g. SBI Mutual Fund)",
    "scheme_name": "Full official AMFI scheme name",
    "category": "Equity / Debt / Hybrid",
    "sub_category": "Large Cap / Mid Cap / Small Cap / Liquid etc.",
    "plan": "Regular or Direct",
    "launch_date": "Fund launch date (YYYY-MM-DD)",
    "benchmark": "Official benchmark index",
    "expense_ratio_pct": "Annual expense ratio in % (0.1–2.5)",
    "exit_load_pct": "Exit load percentage",
    "fund_manager": "Primary fund manager name",
    "risk_category": "SEBI risk: Low / Moderate / High / Very High",
    "sebi_category_code": "Internal SEBI code (EC01, DC01, etc.)",
    "date": "NAV / transaction date (business days only)",
    "nav": "Net Asset Value in Rs.",
    "daily_return": "Computed: nav_t / nav_(t-1) - 1",
    "aum_crore": "AUM in Rs. crore",
    "num_schemes": "Number of schemes under that fund house",
    "month": "Month in YYYY-MM format",
    "sip_inflow_crore": "Total SIP inflows in Rs. crore (AMFI)",
    "active_sip_accounts_crore": "Active SIP accounts in crore",
    "new_sip_accounts_lakh": "New SIP registrations in lakh",
    "sip_aum_lakh_crore": "SIP AUM in Rs. lakh crore",
    "yoy_growth_pct": "YoY growth % in SIP inflows (computed)",
    "net_inflow_crore": "Net inflow into category in Rs. crore",
    "folio_count_crore": "Total folios in crore",
    "return_1yr_pct": "1-year absolute return %",
    "return_3yr_pct": "3-year CAGR %",
    "return_5yr_pct": "5-year CAGR %",
    "sharpe_ratio": "Risk-adjusted return (higher = better, >1 is good)",
    "sortino_ratio": "Like Sharpe but only penalises downside vol",
    "alpha": "Return above benchmark (return_3yr - benchmark_3yr)",
    "beta": "Market sensitivity (1.0 = same as market)",
    "max_drawdown_pct": "Worst peak-to-trough decline (negative)",
    "std_dev_ann_pct": "Annualised std dev of daily returns %",
    "investor_id": "Unique investor ID (INV000001–INV005000)",
    "transaction_type": "SIP / Lumpsum / Redemption",
    "amount_inr": "Transaction amount in Indian Rupees",
    "state": "Investor's Indian state",
    "city_tier": "T30 (Top 30 cities) or B30 (Beyond Top 30)",
    "age_group": "18-25 / 26-35 / 36-45 / 46-55 / 56+",
    "kyc_status": "Verified (~92%) or Pending (~8%)",
    "payment_mode": "UPI / Net Banking / Mandate / Cheque",
    "stock_symbol": "NSE/BSE ticker symbol of holding",
    "weight_pct": "Portfolio weight % of this stock",
    "sector": "Sector of the stock (e.g. Banking, IT)",
    "index_name": "Benchmark index name (e.g. Nifty 50)",
    "close_value": "Closing index value for that date",
}

source_map = {
    "01_fund_master":           "AMFI India / mfapi.in",
    "02_nav_history":           "mfapi.in REST API (real anchor values)",
    "03_aum_by_fund_house":     "AMFI Quarterly Reports",
    "04_monthly_sip_inflows":   "AMFI Monthly Notes",
    "05_category_inflows":      "AMFI Monthly Notes / FY2024-25",
    "06_industry_folio_count":  "AMFI published milestones",
    "07_scheme_performance":    "Computed from nav_history + benchmark_indices",
    "08_investor_transactions": "Synthetic (real geographic/demographic distributions)",
    "09_portfolio_holdings":    "AMFI / Fund factsheets Dec 2025",
    "10_benchmark_indices":     "NSE India / BSE India Bhavcopy",
}

lines = [
    "# Data Dictionary — Bluestock MF Capstone\n",
    "_Auto-generated after cleaning. Reflects processed dataset schemas._\n",
    f"**Total datasets:** {len(datasets)}\n\n---\n"
]

for name, path in datasets.items():
    df_full = pd.read_csv(path)
    df = df_full.head(5)
    lines.append(f"## {name}\n")
    lines.append(f"**Source:** {source_map.get(name, 'See project PDF')}\n")
    lines.append(f"**Rows:** {len(df_full):,}  |  **Columns:** {df.shape[1]}\n\n")
    lines.append("| Column | Dtype | Description |")
    lines.append("|--------|-------|-------------|")
    for col in df.columns:
        dtype = str(df[col].dtype)
        desc = column_descriptions.get(col, "—")
        lines.append(f"| `{col}` | {dtype} | {desc} |")
    lines.append("\n")

output_path = BASE / "data_dictionary.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f" data_dictionary.md saved to {output_path}")

 data_dictionary.md saved to C:\Users\meghr\OneDrive\Desktop\Bluestock\Mutual_funds_Analytis\data_dictionary.md
